<a href="https://colab.research.google.com/github/xavierjacomep/curso-intro-machine-learning/blob/main/titanic_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background:#0F172A; padding:36px 48px;">
  <h1 style="color:#FFFFFF; font-size:2em; margin:0; font-weight:800; letter-spacing:2px;">
    🔧 FEATURE ENGINEERING
  </h1>
  <h2 style="color:#0284C7; font-size:1.1em; margin:12px 0 6px 0; font-weight:400;">
    Caso Práctico: Sobrevivientes del Titanic
  </h2>
  <div style="height:3px; background:#0284C7; width:100px; margin:14px 0;"></div>
  <p style="color:#94A3B8; font-size:0.9em; margin:0;">
    Ejercicio para principiantes · ¿Mejoran los KPIs al crear mejores variables?
  </p>
</div>
<div style="background:#1E293B; padding:16px 48px; margin-bottom:24px;">
  <p style="color:#E2E8F0; margin:0; font-size:0.9em; line-height:1.8;">
    <strong style="color:#0284C7;">Pregunta central:</strong>
    ¿Vale la pena invertir tiempo en construir mejores variables, o con los datos originales es suficiente?<br>
    <strong style="color:#0284C7;">Lo que haremos:</strong>
    Medir KPIs sin Feature Engineering → aplicar 4 técnicas → medir KPIs de nuevo → comparar.<br>
    <strong style="color:#0284C7;">Modelos usados:</strong>
    Regresión Logística y Random Forest — para demostrar que el FE ayuda sin importar el algoritmo.
  </p>
</div>

---
## 📦 Paso 0 — Importar librerías

In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np
import re

# Visualización
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Preprocesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Feature Selection
from sklearn.feature_selection import SelectKBest, chi2, f_classif

# Métricas
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
CV           = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("✅ Librerías cargadas")

---
## 📂 Paso 1 — Cargar datos y pipeline base

Repetimos la limpieza y el encoding mínimo del cuaderno anterior.
Este es el punto de partida: **datos limpios, sin ningún Feature Engineering adicional.**

In [ ]:
df_original = pd.read_csv('train.csv')

# ── Limpieza ──────────────────────────────────────────────────────────
df = df_original.copy()
df = df.drop(columns=['PassengerId', 'Ticket'])   # Cabin la conservamos un momento

# Imputar nulos
df['Age']      = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare']     = df['Fare'].fillna(df['Fare'].median())

# ── Encoding mínimo ───────────────────────────────────────────────────
df['Sex']      = (df['Sex'] == 'female').astype(int)
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# ── Features base (sin ningún FE adicional) ───────────────────────────
FEATURES_BASE = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
TARGET        = 'Survived'

X_base = df[FEATURES_BASE]
y      = df[TARGET]

print(f"Dataset listo: {X_base.shape[0]} filas × {X_base.shape[1]} features base")
print(f"Features base: {FEATURES_BASE}")

---
## 📊 Paso 2 — Baseline: KPIs sin Feature Engineering

Antes de transformar nada, medimos el desempeño de ambos modelos **con los datos tal como están**.
Este número es el piso que debemos superar.

> Usamos Validación Cruzada Estratificada (5 folds) para que la comparación sea justa y estable.

In [ ]:
def evaluar_cv(X, y, cv, label=""):
    """
    Entrena LR y RF con CV y devuelve un dict con media y std
    de Accuracy, Precision, Recall y F1 para cada modelo.
    """
    scaler  = StandardScaler()
    X_sc    = scaler.fit_transform(X)

    modelos = {
        'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
        'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }
    metricas = ['accuracy', 'precision', 'recall', 'f1']
    resultados = {}

    for nombre, modelo in modelos.items():
        # Random Forest no necesita escalado, pero no perjudica usarlo
        res = {}
        for met in metricas:
            scores = cross_val_score(modelo, X_sc, y, cv=cv, scoring=met)
            res[met] = {'media': scores.mean(), 'std': scores.std()}
        resultados[nombre] = res
        print(f"  {nombre:<24}  "
              f"Acc={res['accuracy']['media']:.4f}  "
              f"Prec={res['precision']['media']:.4f}  "
              f"Rec={res['recall']['media']:.4f}  "
              f"F1={res['f1']['media']:.4f}")

    return resultados

print("── BASELINE (sin Feature Engineering) ──────────────────────")
kpis_base = evaluar_cv(X_base, y, CV)
print()
print(f"  Features usados ({len(FEATURES_BASE)}): {FEATURES_BASE}")

---
## 🏗️ Paso 3 — FE Técnica 1: Creación de nuevos features

La primera técnica consiste en **construir variables nuevas combinando o extrayendo información** de las que ya existen.
Los datos originales a veces esconden patrones que el modelo no puede detectar directamente.

### Features que crearemos

| Feature nuevo | De dónde viene | ¿Qué captura? |
|---|---|---|
| `Title` | Extraído del `Name` | Rango social y sexo simultáneamente |
| `FamilySize` | SibSp + Parch + 1 | Tamaño del grupo familiar |
| `IsAlone` | FamilySize == 1 | Si viajaba solo o acompañado |
| `HasCabin` | Presencia en columna `Cabin` | Proxy de clase económica real |

In [ ]:
# ── Feature: Title ───────────────────────────────────────────────────
# El nombre tiene el formato "Apellido, Título. Nombre"
# Extraemos el título y agrupamos los poco frecuentes en 'Rare'

def extraer_titulo(nombre):
    match = re.search(r', ([A-Za-z]+)\.', nombre)
    if not match:
        return 4   # Unknown → Rare
    titulo = match.group(1)
    if titulo == 'Mr':                        return 0
    elif titulo in ('Mrs', 'Mme'):            return 1
    elif titulo in ('Miss', 'Ms', 'Mlle'):    return 2
    elif titulo == 'Master':                  return 3
    else:                                     return 4   # Rare

df['Title'] = df_original['Name'].apply(extraer_titulo)

# Mostrar supervivencia por título para validar que tiene poder predictivo
mapa_titulo = {0: 'Mr', 1: 'Mrs', 2: 'Miss', 3: 'Master', 4: 'Rare'}
print("Supervivencia media por Título:")
print(df.groupby('Title')[TARGET].agg(['mean', 'count'])
        .rename(index=mapa_titulo)
        .rename(columns={'mean': 'Tasa superv.', 'count': 'N'})
        .round(3))

In [ ]:
# ── Feature: FamilySize e IsAlone ────────────────────────────────────
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

print("Supervivencia por tamaño de familia:")
print(df.groupby('FamilySize')[TARGET].agg(['mean', 'count'])
        .rename(columns={'mean': 'Tasa superv.', 'count': 'N'})
        .round(3))
print()
print(f"Viajaban solos  (IsAlone=1): {(df['IsAlone']==1).sum()} pasajeros")
print(f"Viajaban acompañados       : {(df['IsAlone']==0).sum()} pasajeros")

In [ ]:
# ── Feature: HasCabin ────────────────────────────────────────────────
# Cabin tiene 77% de nulos — en lugar de descartarla, la convertimos
# en una variable binaria: ¿tenía cabina asignada o no?
df['HasCabin'] = df_original['Cabin'].notna().astype(int)

print("Supervivencia según si tenía cabina:")
print(df.groupby('HasCabin')[TARGET].agg(['mean', 'count'])
        .rename(index={0: 'Sin cabina', 1: 'Con cabina'})
        .rename(columns={'mean': 'Tasa superv.', 'count': 'N'})
        .round(3))

# Ya podemos eliminar las columnas que usamos para crear estos features
df = df.drop(columns=['Name', 'Cabin'])

FEATURES_NUEVOS_B1 = ['Title', 'FamilySize', 'IsAlone', 'HasCabin']
print()
print(f"✅ Features creados: {FEATURES_NUEVOS_B1}")

---
## 📐 Paso 4 — FE Técnica 2: Transformaciones matemáticas

Algunas variables numéricas tienen **distribuciones muy sesgadas**: la mayoría de valores están
concentrados en un rango pequeño y unos pocos valores extremos distorsionan el aprendizaje.

La solución es aplicar una transformación que **comprima** esa distribución y la haga más simétrica.

### ¿Por qué `log(Fare + 1)`?

- `Fare` va de 0 a 512, pero el 75% de pasajeros pagó menos de 31 libras.
- Unos pocos pagaron 500+, creando una "cola larga" que distorsiona al modelo.
- `log(x + 1)` comprime esa cola sin perder información (el +1 evita log(0)).

In [ ]:
# Visualizar la distribución antes y después
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(df['Fare'], bins=40, color='#94A3B8', edgecolor='white')
axes[0].set_title('Fare — ORIGINAL\n(distribución muy sesgada)', fontweight='bold')
axes[0].set_xlabel('Tarifa (libras)')
axes[0].set_ylabel('Cantidad de pasajeros')
axes[0].axvline(df['Fare'].mean(),   color='#DC2626', linestyle='--', linewidth=1.5, label=f"Media = {df['Fare'].mean():.1f}")
axes[0].axvline(df['Fare'].median(), color='#0284C7', linestyle='--', linewidth=1.5, label=f"Mediana = {df['Fare'].median():.1f}")
axes[0].legend(fontsize=9)

df['Fare_log'] = np.log1p(df['Fare'])   # log1p(x) = log(x + 1)

axes[1].hist(df['Fare_log'], bins=40, color='#0284C7', edgecolor='white')
axes[1].set_title('Fare_log — TRANSFORMADO\n(distribución más simétrica)', fontweight='bold')
axes[1].set_xlabel('log(Tarifa + 1)')
axes[1].axvline(df['Fare_log'].mean(),   color='#DC2626', linestyle='--', linewidth=1.5, label=f"Media = {df['Fare_log'].mean():.2f}")
axes[1].axvline(df['Fare_log'].median(), color='#0284C7', linestyle='--', linewidth=1.5, label=f"Mediana = {df['Fare_log'].median():.2f}")
axes[1].legend(fontsize=9)

plt.suptitle('Transformación Logarítmica de Fare', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

FEATURES_NUEVOS_B2 = ['Fare_log']
print(f"✅ Feature creado: {FEATURES_NUEVOS_B2}")
print(f"   Skewness antes: {df['Fare'].skew():.3f}  →  después: {df['Fare_log'].skew():.3f}")
print("   (Skewness más cercano a 0 = distribución más simétrica)")

---
## 🗂️ Paso 5 — FE Técnica 3: Binning (discretización)

El **binning** convierte una variable numérica continua en grupos (bins).

¿Por qué hacerlo? Porque a veces la relación entre una variable y el target no es lineal.
Por ejemplo, la supervivencia no sube constantemente con la edad — los niños pequeños
tuvieron prioridad independientemente de su edad exacta.

Agrupar edades en rangos permite que el modelo capture ese patrón más fácilmente.

| Feature | Bins | Lógica |
|---|---|---|
| `AgeBand` | Niño / Joven / Adulto / Mayor | Grupos con patrones de evacuación distintos |
| `FareBand` | 4 cuartiles (Q1-Q4) | Proxy de nivel económico relativo |

In [ ]:
# ── AgeBand ───────────────────────────────────────────────────────────
df['AgeBand'] = pd.cut(
    df['Age'],
    bins=[0, 12, 25, 60, 100],
    labels=[0, 1, 2, 3]          # 0=Niño, 1=Joven, 2=Adulto, 3=Mayor
).astype(int)

# Visualizar supervivencia por banda de edad
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

etiquetas_age  = ['Niño\n(0-12)', 'Joven\n(12-25)', 'Adulto\n(25-60)', 'Mayor\n(60+)']
surv_age = df.groupby('AgeBand', observed=True)[TARGET].mean() * 100
axes[0].bar(etiquetas_age, surv_age.values, color='#0284C7', width=0.5, edgecolor='white')
axes[0].set_title('Supervivencia por AgeBand', fontweight='bold')
axes[0].set_ylabel('Tasa de supervivencia (%)')
axes[0].set_ylim(0, 75)
for i, v in enumerate(surv_age.values):
    axes[0].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)

# ── FareBand ──────────────────────────────────────────────────────────
df['FareBand'] = pd.qcut(df['Fare'], q=4, labels=[0, 1, 2, 3]).astype(int)

etiquetas_fare = ['Q1\n(más bajo)', 'Q2', 'Q3', 'Q4\n(más alto)']
surv_fare = df.groupby('FareBand', observed=True)[TARGET].mean() * 100
axes[1].bar(etiquetas_fare, surv_fare.values, color='#0F766E', width=0.5, edgecolor='white')
axes[1].set_title('Supervivencia por FareBand (cuartiles)', fontweight='bold')
axes[1].set_ylabel('Tasa de supervivencia (%)')
axes[1].set_ylim(0, 75)
for i, v in enumerate(surv_fare.values):
    axes[1].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Binning: los grupos muestran patrones claros en supervivencia',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

FEATURES_NUEVOS_B3 = ['AgeBand', 'FareBand']
print(f"✅ Features creados: {FEATURES_NUEVOS_B3}")

---
## 🎯 Paso 6 — FE Técnica 4: Selección de features con SelectKBest

Ahora tenemos **más features de los que empezamos**. Pero más no siempre es mejor:
los features irrelevantes o redundantes pueden añadir ruido y perjudicar al modelo.

`SelectKBest` evalúa cada feature de forma **estadística** midiendo su relación con el target
y selecciona los K más relevantes.

### ¿Qué test estadístico usamos?
Usamos **f_classif** (test F de ANOVA): mide si la media de un feature difiere significativamente
entre las dos clases (sobrevivió vs no sobrevivió). Funciona con variables numéricas continuas
y no requiere que los datos sean positivos (a diferencia de chi²).

### Pool completo para selección
SelectKBest evaluará **todos los features** — originales + creados en B1, B2 y B3 —
y elegirá los K mejores. Así el selector tiene la máxima información disponible.

In [ ]:
# Pool completo: features base + todos los nuevos creados
POOL_COMPLETO = FEATURES_BASE + FEATURES_NUEVOS_B1 + FEATURES_NUEVOS_B2 + FEATURES_NUEVOS_B3
# Eliminar duplicados manteniendo el orden
POOL_COMPLETO = list(dict.fromkeys(POOL_COMPLETO))

X_pool = df[POOL_COMPLETO]

print(f"Pool completo de features para selección ({len(POOL_COMPLETO)}):")
for f in POOL_COMPLETO:
    print(f"  · {f}")

In [ ]:
# Aplicar SelectKBest con f_classif
# Necesitamos escalar antes porque f_classif trabaja mejor con variables en escala comparable
scaler_sel = StandardScaler()
X_pool_sc  = scaler_sel.fit_transform(X_pool)

selector = SelectKBest(score_func=f_classif, k='all')   # k='all' para ver todos los scores primero
selector.fit(X_pool_sc, y)

# Tabla de scores por feature
scores_df = pd.DataFrame({
    'Feature':  POOL_COMPLETO,
    'F-Score':  selector.scores_.round(2),
    'P-valor':  selector.pvalues_.round(4),
}).sort_values('F-Score', ascending=False).reset_index(drop=True)

scores_df['Relevante'] = scores_df['P-valor'].apply(lambda p: '✅' if p < 0.05 else '❌')

print("Ranking de features por F-Score (test ANOVA):")
print("(F-Score alto + p-valor < 0.05 = feature estadísticamente relevante)")
print()
print(scores_df.to_string(index=False))

In [ ]:
# Visualizar los F-Scores
fig, ax = plt.subplots(figsize=(9, 5))

features_sorted = scores_df['Feature'].tolist()
fscores_sorted  = scores_df['F-Score'].tolist()
relevante       = scores_df['Relevante'].tolist()

colores = ['#0284C7' if r == '✅' else '#CBD5E1' for r in relevante]
bars = ax.barh(features_sorted[::-1], fscores_sorted[::-1],
               color=colores[::-1], height=0.6, edgecolor='white')

ax.set_xlabel('F-Score (ANOVA)')
ax.set_title('Relevancia de cada feature para predecir Survived\n'
             '(azul = estadísticamente significativo  p < 0.05)',
             fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, fscores_sorted[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

patch_rel  = mpatches.Patch(color='#0284C7', label='p < 0.05 (relevante)')
patch_nrel = mpatches.Patch(color='#CBD5E1', label='p ≥ 0.05 (no significativo)')
ax.legend(handles=[patch_rel, patch_nrel], fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Seleccionar los K mejores features
# Criterio: todos los que son estadísticamente significativos (p < 0.05)
# Si hubiera muchos, limitaríamos a K=10 o similar
features_significativos = scores_df[scores_df['P-valor'] < 0.05]['Feature'].tolist()
K = len(features_significativos)

print(f"Features seleccionados (p < 0.05): {K} de {len(POOL_COMPLETO)}")
print()
for f in features_significativos:
    row = scores_df[scores_df['Feature'] == f].iloc[0]
    print(f"  ✅ {f:<14}  F={row['F-Score']:>7.2f}  p={row['P-valor']:.4f}")

# Aplicar la selección
selector_final = SelectKBest(score_func=f_classif, k=K)
X_fe           = pd.DataFrame(
    selector_final.fit_transform(X_pool_sc, y),
    columns=features_significativos
)

FEATURES_FINALES = features_significativos
print()
print(f"✅ Dataset final con Feature Engineering: {X_fe.shape[0]} filas × {X_fe.shape[1]} features")

---
## 📊 Paso 7 — KPIs con Feature Engineering

Ahora medimos el desempeño de los mismos dos modelos, con las mismas condiciones
(CV 5-fold estratificada), pero usando el dataset enriquecido con FE.

> Los datos ya están escalados desde el Paso 6 (SelectKBest los recibió escalados),
> por lo que los usamos directamente.

In [ ]:
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}
metricas_nombre = ['accuracy', 'precision', 'recall', 'f1']
kpis_fe = {}

print("── CON Feature Engineering ─────────────────────────────────")
for nombre, modelo in modelos.items():
    res = {}
    for met in metricas_nombre:
        scores = cross_val_score(modelo, X_fe, y, cv=CV, scoring=met)
        res[met] = {'media': scores.mean(), 'std': scores.std()}
    kpis_fe[nombre] = res
    print(f"  {nombre:<24}  "
          f"Acc={res['accuracy']['media']:.4f}  "
          f"Prec={res['precision']['media']:.4f}  "
          f"Rec={res['recall']['media']:.4f}  "
          f"F1={res['f1']['media']:.4f}")

print()
print(f"  Features usados ({len(FEATURES_FINALES)}): {FEATURES_FINALES}")

---
## 📈 Paso 8 — Comparación visual: antes vs después del Feature Engineering

Ahora ponemos los dos escenarios frente a frente para responder la pregunta central:
**¿mejoró el desempeño de los modelos al aplicar Feature Engineering?**

In [ ]:
# ── Gráfico 1: comparación por modelo y por métrica ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nombres_modelos  = list(modelos.keys())
nombres_metricas = ['accuracy', 'precision', 'recall', 'f1']
etiq_metricas    = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x      = np.arange(len(etiq_metricas))
ancho  = 0.35

colores_base = ['#94A3B8', '#CBD5E1']
colores_fe   = ['#0284C7', '#0F766E']

for idx, (nombre_modelo, ax) in enumerate(zip(nombres_modelos, axes)):
    vals_base = [kpis_base[nombre_modelo][m]['media'] for m in nombres_metricas]
    vals_fe   = [kpis_fe[nombre_modelo][m]['media']   for m in nombres_metricas]
    stds_fe   = [kpis_fe[nombre_modelo][m]['std']     for m in nombres_metricas]

    bars1 = ax.bar(x - ancho/2, vals_base, width=ancho,
                   color=colores_base[idx], label='Sin FE', edgecolor='white')
    bars2 = ax.bar(x + ancho/2, vals_fe, width=ancho,
                   color=colores_fe[idx], label='Con FE', edgecolor='white',
                   yerr=stds_fe, capsize=4,
                   error_kw={'elinewidth': 1.8, 'ecolor': '#0F172A'})

    # Etiquetas de valor
    for bar, val in zip(bars1, vals_base):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', fontsize=9, color='#475569', fontweight='bold')
    for bar, val in zip(bars2, vals_fe):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.028,
                f'{val:.3f}', ha='center', fontsize=9, color=colores_fe[idx], fontweight='bold')

    # Flechas de mejora
    for i, (vb, vf) in enumerate(zip(vals_base, vals_fe)):
        diff = vf - vb
        if diff > 0.001:
            ax.annotate('', xy=(i + ancho/2, vf + 0.045),
                        xytext=(i + ancho/2, vf + 0.025),
                        arrowprops=dict(arrowstyle='->', color='#16A34A', lw=1.5))

    ax.set_xticks(x)
    ax.set_xticklabels(etiq_metricas, fontsize=10)
    ax.set_ylim(0.55, 1.08)
    ax.set_ylabel('Score (CV 5-fold media)')
    ax.set_title(nombre_modelo, fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Sin Feature Engineering vs Con Feature Engineering\n'
             '(barras de error = ±1 std de la CV · flechas verdes = mejora)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: mapa de calor de mejoras ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))

filas   = nombres_modelos
cols    = etiq_metricas
mejoras = np.array([
    [kpis_fe[m][met]['media'] - kpis_base[m][met]['media']
     for met in nombres_metricas]
    for m in nombres_modelos
])

im = ax.imshow(mejoras, cmap='RdYlGn', aspect='auto', vmin=-0.05, vmax=0.10)
plt.colorbar(im, ax=ax, label='Δ Score (FE − Base)')

ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(filas)))
ax.set_xticklabels(cols, fontsize=11)
ax.set_yticklabels(filas, fontsize=11)
ax.set_title('Mapa de mejoras por técnica de Feature Engineering\n'
             '(verde = mejora · rojo = empeora · blanco = sin cambio)',
             fontweight='bold')

for i in range(len(filas)):
    for j in range(len(cols)):
        signo = '+' if mejoras[i, j] >= 0 else ''
        ax.text(j, i, f'{signo}{mejoras[i, j]:.4f}',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='#0F172A')

plt.tight_layout()
plt.show()

---
## ✅ Paso 9 — Conclusión: ¿qué aportó cada técnica?

In [ ]:
# ── Tabla resumen final ───────────────────────────────────────────────
print("=" * 68)
print("  RESUMEN FINAL: KPIs ANTES vs DESPUÉS DEL FEATURE ENGINEERING")
print("=" * 68)

for nombre_modelo in nombres_modelos:
    print(f"\n  {nombre_modelo}")
    print(f"  {'Métrica':<12} {'Sin FE':>10} {'Con FE':>10} {'Δ':>10}  {'Veredicto'}")
    print("  " + "-" * 52)
    for met, etiq in zip(nombres_metricas, etiq_metricas):
        vb   = kpis_base[nombre_modelo][met]['media']
        vf   = kpis_fe[nombre_modelo][met]['media']
        diff = vf - vb
        if diff > 0.005:
            veredicto = f"▲ +{diff:.4f}  mejora"
        elif diff < -0.005:
            veredicto = f"▼  {diff:.4f}  empeora"
        else:
            veredicto = f"  ≈ {diff:.4f}  sin cambio"
        print(f"  {etiq:<12} {vb:>10.4f} {vf:>10.4f} {veredicto}")

print()
print("=" * 68)

In [ ]:
# ── Feature más relevante según SelectKBest ──────────────────────────
print("Ranking de features por relevancia estadística (F-Score ANOVA):")
print()
for i, row in scores_df.iterrows():
    bar = '█' * int(row['F-Score'] / scores_df['F-Score'].max() * 20)
    print(f"  {i+1:>2}. {row['Feature']:<14} {bar:<20} F={row['F-Score']:>7.2f}  {row['Relevante']}")

print()
print(f"  Feature más relevante: {scores_df.iloc[0]['Feature']}")
print(f"  Feature menos relevante (seleccionado): {scores_df[scores_df['Relevante']=='✅'].iloc[-1]['Feature']}")

---
### 📌 Lecciones aprendidas

| Técnica | ¿Qué hicimos? | ¿Por qué funciona? |
|---|---|---|
| **Creación** | `Title`, `FamilySize`, `IsAlone`, `HasCabin` | Captura relaciones que el modelo no puede inferir solo de las variables originales |
| **Transformación log** | `Fare_log` | Reduce el efecto de outliers y simetriza la distribución |
| **Binning** | `AgeBand`, `FareBand` | Convierte relaciones no lineales en patrones más fáciles de aprender |
| **SelectKBest** | Filtrado del pool completo | Elimina ruido: features irrelevantes perjudican al modelo |

> **Conclusión central:** el Feature Engineering no cambia los datos reales —
> cambia **cómo el modelo los ve**. Un modelo con datos bien preparados
> casi siempre supera a un modelo más complejo con datos crudos.